# Coleta das ações: NVIDIA (NVDA) e Nasdaq-100 (QQQ)
Pregões de 19/10 a 18/11/2026, o mesmo recorte do `coleta_youtube.ipynb` (18/10 é domingo). Uma linha por pregão. A coluna `data_pregao` é a chave para juntar com o YouTube e com as notícias. Fonte: Yahoo Finance, pela biblioteca `yfinance` (`pip install yfinance`).

In [1]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\eaime\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import os
import pathlib
import numpy as np
import pandas as pd
import yfinance as yf

# Mesmo recorte do coleta_youtube.ipynb: pregões de 19/10 a 18/11/2026.
INICIO, FIM = "2026-10-19", "2026-11-18"
BUFFER_DIAS = 7                            # dias de calendário antes do INICIO, só para calcular a variação do 1º pregão
TICKERS = {"nvda": "NVDA", "qqq": "QQQ"}   # prefixo das colunas -> ticker (QQQ = Nasdaq-100, para descontar o mercado)
FERIADOS_BOLSA = []                        # feriados da NYSE (AAAA-MM-DD); nenhum entre 19/10 e 18/11/2026. Igual ao do YouTube.
ARQUIVO_MANUAL = {}                        # plano B: nomes de arquivos dentro de data/raw, ex.: {"nvda": "NVDA.csv", "qqq": "QQQ.csv"}

def raiz_do_projeto():
    # Acha a raiz do repositório (a pasta com .git), de onde quer que o notebook rode, em qualquer computador.
    candidatos = []
    if os.environ.get("PROJETO_DIR"):                 # opcional: força o caminho pela variável de ambiente PROJETO_DIR
        candidatos.append(pathlib.Path(os.environ["PROJETO_DIR"]))
    arq = globals().get("__vsc_ipynb_file__")         # caminho do próprio notebook, quando aberto no VS Code
    if arq:
        candidatos.append(pathlib.Path(arq).parent)
    candidatos.append(pathlib.Path.cwd())             # pasta de execução do kernel
    for c in candidatos:
        try:
            c = c.resolve()
        except OSError:
            continue
        for d in [c, *c.parents]:
            if (d / ".git").exists():
                return d
    raise RuntimeError("Não achei a raiz do projeto (pasta com .git). Abra o notebook de dentro do repositório "
                       f"(pasta notebooks/) ou defina PROJETO_DIR. Tentei: {[str(c) for c in candidatos]}")


RAIZ = raiz_do_projeto()
RAW, PROC = RAIZ / "data" / "raw", RAIZ / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", RAIZ)

# Calendário esperado de pregões (o mesmo critério do notebook do YouTube).
PREGOES = pd.bdate_range(INICIO, FIM, freq="C", holidays=FERIADOS_BOLSA)
print(len(PREGOES), "pregões esperados:", PREGOES[0].date(), "->", PREGOES[-1].date())

Raiz do projeto: C:\Users\eaime\Downloads\jensen-huang-nvidia-analysis
23 pregões esperados: 2026-10-19 -> 2026-11-18


## Coleta
Baixa NVDA e QQQ de 7 dias antes do início até 18/11. O `end` do yfinance não é incluído, então o código soma um dia. A janela extra serve só para calcular a variação do primeiro pregão. Os preços são ajustados (`auto_adjust=True`) e os dados brutos ficam em `data/raw/acoes_bruto_*.csv`.

**Plano B:** o yfinance não é uma API oficial e o Yahoo muda o site de vez em quando. Se a coleta falhar, baixe o CSV de cada ticker (Yahoo Finance, aba *Historical Data*, de 12/10 a 18/11), salve em `data/raw/` e coloque o nome do arquivo em `ARQUIVO_MANUAL`. O CSV precisa ter as colunas `Date`, `Open`, `High`, `Low`, `Close` e `Volume`.

In [3]:
def baixar(ticker):
    ini = (pd.Timestamp(INICIO) - pd.Timedelta(days=BUFFER_DIAS)).strftime("%Y-%m-%d")
    fim_exclusivo = (pd.Timestamp(FIM) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    h = yf.Ticker(ticker).history(start=ini, end=fim_exclusivo, auto_adjust=True)
    if h.empty:
        raise RuntimeError(f"{ticker}: o Yahoo não devolveu dados. Tente de novo ou use o plano B (ARQUIVO_MANUAL).")
    h.index = pd.DatetimeIndex(h.index).tz_localize(None).normalize()      # só a data do pregão
    return h[["Open", "High", "Low", "Close", "Volume"]]


def ler_manual(caminho):
    m = pd.read_csv(caminho)
    m.columns = [c.strip().title() for c in m.columns]
    m["Date"] = pd.to_datetime(m["Date"])
    return m.set_index("Date").sort_index()[["Open", "High", "Low", "Close", "Volume"]]


bruto = {}
for pref, tk in TICKERS.items():
    if pref in ARQUIVO_MANUAL:
        bruto[pref] = ler_manual(RAW / ARQUIVO_MANUAL[pref])
        origem = f"arquivo manual {ARQUIVO_MANUAL[pref]}"
    else:
        bruto[pref] = baixar(tk)
        origem = "yfinance"
    bruto[pref].index.name = "data"
    bruto[pref].to_csv(RAW / f"acoes_bruto_{tk.lower()}.csv")
    h = bruto[pref]
    print(f"{tk}: {len(h)} linhas, {h.index.min().date()} -> {h.index.max().date()} ({origem})")

NVDA: 28 linhas, 2026-10-12 -> 2026-11-18 (yfinance)
QQQ: 28 linhas, 2026-10-12 -> 2026-11-18 (yfinance)


## Variações e retorno anormal
- `*_variacao_pct`: fechamento ÷ fechamento do pregão anterior − 1, em %.
- `nvda_gap_abertura_pct`: abertura ÷ fechamento anterior − 1, em %. Mostra a reação a notícias que saíram com a bolsa fechada (fim de semana ou depois das 16h), que é exatamente o que `data_pregao` desloca para o pregão seguinte.
- `retorno_anormal_pp`: variação da NVDA menos a do QQQ, em pontos percentuais. É uma correção simples, sem beta, para separar o que foi do mercado inteiro. Como a NVDA faz parte do QQQ, ela pode subestimar um pouco a diferença.
- `nvda_volume`: ações negociadas no dia, que serve como medida de atenção do mercado.

As variações são calculadas antes do corte de período, então o primeiro pregão (19/10) usa o fechamento de 16/10. O código confere se os pregões que vieram do Yahoo são exatamente os do calendário esperado: se faltar ou sobrar algum dia, ele para, porque um calendário diferente do YouTube e das notícias estragaria o cruzamento.

In [13]:
def preparar(h, pref, completo):
    h = h.sort_index()
    h = h[~h.index.duplicated(keep="last")]
    out = pd.DataFrame(index=h.index)
    out[f"{pref}_fechamento"] = h["Close"]
    out[f"{pref}_variacao_pct"] = h["Close"].pct_change() * 100
    if completo:
        out[f"{pref}_gap_abertura_pct"] = (h["Open"] / h["Close"].shift(1) - 1) * 100
        out[f"{pref}_volume"] = h["Volume"]
    return out


acoes = preparar(bruto["nvda"], "nvda", completo=True).join(preparar(bruto["qqq"], "qqq", completo=False), how="outer")
acoes["retorno_anormal_pp"] = acoes["nvda_variacao_pct"] - acoes["qqq_variacao_pct"]

# O calendário do Yahoo tem de ser igual ao esperado.
no_periodo = acoes.index[(acoes.index >= pd.Timestamp(INICIO)) & (acoes.index <= pd.Timestamp(FIM))]
faltando, sobrando = PREGOES.difference(no_periodo), no_periodo.difference(PREGOES)
if len(faltando) or len(sobrando):
    raise RuntimeError(f"Calendário diferente do esperado. Faltam: {[d.date().isoformat() for d in faltando]} | "
                       f"sobram: {[d.date().isoformat() for d in sobrando]}. Se for feriado da bolsa, ajuste FERIADOS_BOLSA "
                       "aqui e no notebook do YouTube.")

acoes = acoes.loc[PREGOES]
acoes.index.name = "data_pregao"
for c in [c for c in acoes.columns if c.endswith(("_pct", "_pp"))]:
    acoes[c] = acoes[c].round(4)
acoes = acoes.reset_index()
acoes["data_pregao"] = acoes["data_pregao"].dt.date
acoes["data_coleta"] = pd.Timestamp.now(tz="America/New_York").date()

acoes.to_csv(PROC / "acoes_nvidia.csv", index=False)
print(len(acoes), "pregões salvos em", PROC / "acoes_nvidia.csv")

23 pregões salvos em C:\Users\eaime\Downloads\jensen-huang-nvidia-analysis\data\processed\acoes_nvidia.csv


## Conferência
Confira o primeiro e o último pregão e os dias de maior queda e alta da NVDA. O `retorno_anormal_pp` mostra quanto do movimento foi além do mercado.

In [5]:
print(acoes[["data_pregao", "nvda_fechamento"]].iloc[[0, -1]].to_string(index=False))
print("\nValores ausentes:", int(acoes.drop(columns="data_coleta").isna().sum().sum()))

colunas = ["data_pregao", "nvda_variacao_pct", "qqq_variacao_pct", "retorno_anormal_pp", "nvda_gap_abertura_pct", "nvda_volume"]
print("\n5 maiores quedas da NVDA:")
print(acoes.nsmallest(5, "nvda_variacao_pct")[colunas].to_string(index=False))
print("\n5 maiores altas da NVDA:")
print(acoes.nlargest(5, "nvda_variacao_pct")[colunas].to_string(index=False))

data_pregao  nvda_fechamento
 2026-10-19       217.089996
 2026-11-18       150.960007

Valores ausentes: 0

5 maiores quedas da NVDA:
data_pregao  nvda_variacao_pct  qqq_variacao_pct  retorno_anormal_pp  nvda_gap_abertura_pct  nvda_volume
 2026-11-05            -9.2590           -6.4889             -2.7701                -2.0287    306127800
 2026-11-10            -5.4738           -0.6756             -4.7981                -2.6305    429273200
 2026-10-22            -4.3970           -2.8585             -1.5385                -2.1805    125407600
 2026-11-12            -4.3401           -0.3201             -4.0199               -13.0778    747707600
 2026-11-09            -4.2703            3.5437             -7.8139                -7.2707    462719300

5 maiores altas da NVDA:
data_pregao  nvda_variacao_pct  qqq_variacao_pct  retorno_anormal_pp  nvda_gap_abertura_pct  nvda_volume
 2026-11-11             2.8658            2.3149              0.5509                 2.3151    339214800

In [6]:
acoes

,data_pregao,nvda_fechamento,nvda_variacao_pct,nvda_gap_abertura_pct,nvda_volume,qqq_fechamento,qqq_variacao_pct,retorno_anormal_pp,data_coleta
0,2026-10-19,217.089996,1.3823,1.8540,98684700,718.359985,1.2060,0.1764,2026-11-20
1,2026-10-20,216.470001,-0.2856,-0.5113,79609100,717.630005,-0.1016,-0.1840,2026-11-20
2,2026-10-21,221.970001,2.5408,2.0742,87085700,725.200012,1.0549,1.4859,2026-11-20
3,2026-10-22,212.210007,-4.3970,-2.1805,125407600,704.469971,-2.8585,-1.5385,2026-11-20
4,2026-10-23,210.960007,-0.5890,-0.5749,80189800,704.400024,-0.0099,-0.5791,2026-11-20
5,2026-10-26,216.000000,2.3891,1.0191,90829300,712.200012,1.1073,1.2818,2026-11-20
6,2026-10-27,217.940002,0.8981,1.0694,84799700,721.250000,1.2707,-0.3726,2026-11-20
7,2026-10-28,215.380005,-1.1746,0.9085,93030200,725.609985,0.6045,-1.7791,2026-11-20
8,2026-10-29,213.660004,-0.7986,-0.4736,77568100,715.340027,-1.4154,0.6168,2026-11-20
9,2026-10-30,204.639999,-4.2217,-2.2091,168386400,690.489990,-3.4739,-0.7478,2026-11-20


## Validação do CSV salvo
Lê o arquivo que será entregue e confere datas, valores ausentes, o cálculo das variações e o retorno anormal. Os três dias sorteados servem para conferir o fechamento à mão no Yahoo Finance (o preço é ajustado, então pode diferir um pouco do preço "cru" de outras fontes).

In [7]:
salvo = pd.read_csv(PROC / "acoes_nvidia.csv", parse_dates=["data_pregao"])
numericas = salvo.drop(columns=["data_pregao", "data_coleta"])

print("pregões:", len(salvo), "| datas duplicadas:", int(salvo["data_pregao"].duplicated().sum()))
print("datas iguais ao calendário esperado:", list(salvo["data_pregao"]) == list(PREGOES))
print("valores ausentes:", int(numericas.isna().sum().sum()))
print("fechamentos <= 0:", int((salvo[["nvda_fechamento", "qqq_fechamento"]] <= 0).sum().sum()))
print("data_pregao sempre em dia útil:", bool((salvo["data_pregao"].dt.dayofweek < 5).all()))

for pref in ("nvda", "qqq"):   # a partir do 2º pregão, a variação salva tem de bater com os fechamentos salvos
    recalc = salvo[f"{pref}_fechamento"].pct_change() * 100
    ok = np.allclose(recalc.iloc[1:], salvo[f"{pref}_variacao_pct"].iloc[1:], atol=1e-3)
    print(f"{pref}: variação confere com os fechamentos:", ok)
print("retorno_anormal = NVDA - QQQ:", np.allclose(salvo["retorno_anormal_pp"], salvo["nvda_variacao_pct"] - salvo["qqq_variacao_pct"], atol=1e-3))

for _, r in salvo.sample(3, random_state=1).iterrows():
    print(r["data_pregao"].date(), "| NVDA fechou em", round(r["nvda_fechamento"], 2), "| variação", r["nvda_variacao_pct"], "%")

pregões: 23 | datas duplicadas: 0
datas iguais ao calendário esperado: True
valores ausentes: 0
fechamentos <= 0: 0
data_pregao sempre em dia útil: True
nvda: variação confere com os fechamentos: True
qqq: variação confere com os fechamentos: True
retorno_anormal = NVDA - QQQ: True
2026-11-16 | NVDA fechou em 156.14 | variação -4.1204 %
2026-11-11 | NVDA fechou em 173.73 | variação 2.8658 %
2026-10-22 | NVDA fechou em 212.21 | variação -4.397 %


## Teste de junção com o YouTube
Junta pela `data_pregao`. O YouTube tem um pregão a mais (19/11, com `cobertura_completa` falso), que sai na junção. Só roda se `youtube_diario_pregao.csv` existir.

In [8]:
arq_yt = PROC / "youtube_diario_pregao.csv"
if arq_yt.exists():
    yt = pd.read_csv(arq_yt, parse_dates=["data_pregao"])
    j = salvo.merge(yt, on="data_pregao", how="inner", suffixes=("", "_yt"))
    print("pregões em comum:", len(j), "de", len(salvo), "das ações e", len(yt), "do YouTube")
    print("só no YouTube:", sorted(d.date().isoformat() for d in set(yt["data_pregao"]) - set(salvo["data_pregao"])))
    print("só nas ações:", sorted(d.date().isoformat() for d in set(salvo["data_pregao"]) - set(yt["data_pregao"])))
    print("pregões com cobertura completa do YouTube:", int(j["cobertura_completa"].sum()))
else:
    print("youtube_diario_pregao.csv ainda não existe: rode o coleta_youtube.ipynb primeiro.")

pregões em comum: 23 de 23 das ações e 24 do YouTube
só no YouTube: ['2026-11-19']
só nas ações: []
pregões com cobertura completa do YouTube: 22


## Documentação da fonte e limitações (para o relatório)
**Fonte:** Yahoo Finance, acessado pela biblioteca `yfinance` na data da coluna `data_coleta`. Tickers: NVDA (NVIDIA) e QQQ (ETF do Nasdaq-100, usado como referência do mercado). Período: pregões de 19/10 a 18/11/2026, dias da bolsa de Nova York.

**Por que junto com as outras bases:** `data_pregao` é a mesma chave das tabelas do YouTube e das notícias, com a mesma regra de calendário: publicações de fim de semana ou depois das 16h (NY) caem no pregão seguinte.

**Limitações:**
1. O `yfinance` não é uma API oficial do Yahoo. Os valores podem diferir um pouco dos de outras fontes, e a coleta pode quebrar se o Yahoo mudar algo (por isso o plano B).
2. Os preços são ajustados por dividendos e desdobramentos (`auto_adjust=True`). Para o período do estudo isso muda pouco, mas o fechamento pode não coincidir com o "preço cru" de outros sites.
3. Os dados são diários. Não dá para separar o que aconteceu antes e depois de uma notícia dentro do mesmo pregão. `nvda_gap_abertura_pct` ajuda só com as notícias que saíram com a bolsa fechada.
4. `retorno_anormal_pp` é a diferença simples entre NVDA e QQQ, sem ajuste por beta. Outros fatos do mesmo dia (resultados de empresas, dados econômicos, notícias da China) também mexem na ação.
5. São cerca de 23 pregões, e a maior parte das variações grandes está em poucos dias. Isso limita o que a análise estatística pode afirmar: os resultados descrevem uma relação observada neste período, não causa.